# ADRC cohort preprocessing validation: V1 vs V2

Validation follows the preprocessing order. Existing preprocessing outputs are read; only visualization derivatives that are not already present are generated. Missing inputs are skipped and recorded. No preprocessing is rerun.

## 1. Configuration and helpers

In [ ]:
from pathlib import Path
import shutil, subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
SUBJECTS=['D0007','D0051','D0057','D0066','D0089','D0064','D0083','D0087','D0106','D0128']
V1_ROOT=Path('/mnt/newStor/paros/paros_WORK/xintong/test/preprocess_v1_qc')
V2_ROOT=Path('/mnt/newStor/paros/paros_WORK/xintong/test/preprocess_v2')
QC_ROOT=Path('/mnt/newStor/paros/paros_WORK/xintong/test/cohort_preprocessing_validation_ordered')
FREEVIEW_ROOT=QC_ROOT/'freeview_images'; REPORT_ROOT=QC_ROOT/'eddy_quad_reports'
QC_ROOT.mkdir(parents=True,exist_ok=True); FREEVIEW_ROOT.mkdir(parents=True,exist_ok=True); REPORT_ROOT.mkdir(parents=True,exist_ok=True)
DWI_VOLUME=2; RUN_EDDY_QUAD=True
def run_optional(cmd,out=None):
    if out is not None and Path(out).exists(): return 'exists'
    if shutil.which(str(cmd[0])) is None: return f'missing command: {cmd[0]}'
    r=subprocess.run([str(x) for x in cmd],capture_output=True,text=True)
    return 'completed' if r.returncode==0 else f'failed: {r.stderr[-300:]}'
def load_numbers(path,skiprows=0):
    if path is None or not Path(path).exists() or Path(path).stat().st_size==0: return None
    try: return np.loadtxt(path,skiprows=skiprows,ndmin=2)
    except (OSError,ValueError): return None

## 2. Acquisition metadata

This section reads the actual preserved Eddy configuration. `eddy_config.txt`, reverse-PE pair evidence, and the recorded Eddy command are used; no phase-encoding mode is inferred from filenames alone.

In [ ]:
def complete_eddy_dir(root):
    required=['dwi_post_eddy.nii.gz','eddy_indices.txt','eddy_config.txt','eddy_mask.nii','bvals']
    candidates=sorted((Path(root)/'eddy_scratch').glob('dwifslpreproc-tmp-*')) if (Path(root)/'eddy_scratch').exists() else []
    good=[d for d in candidates if all((d/f).exists() for f in required)]
    return good[-1] if good else None
metadata=[]
for subject in SUBJECTS:
    for pipeline,root in [('v1',V1_ROOT/subject),('v2',V2_ROOT/subject)]:
        d=complete_eddy_dir(root); command=d/'dwi_post_eddy.eddy_command_txt' if d and (d/'dwi_post_eddy.eddy_command_txt').exists() else None
        text=command.read_text() if command else ''
        metadata.append({'subject':subject,'pipeline':pipeline,'status':'available' if d else 'missing','phase_encoding_config':(d/'eddy_config.txt').read_text().strip() if d else '','reverse_pe_pair':(root/f'{subject}_se_epi_pair.nii.gz').exists() or (root/f'{subject}_rev_seepi_pair.nii.gz').exists(),'eddy_mode':'rpe_pair/topup' if '--topup=field' in text else 'other/unknown','actual_command':text.strip()})
metadata=pd.DataFrame(metadata); metadata.to_csv(QC_ROOT/'acquisition_metadata.csv',index=False); display(metadata[['subject','pipeline','status','phase_encoding_config','reverse_pe_pair','eddy_mode']])

## 3. Denoising QC

The preprocessing outputs are read. The missing visualization derivative `raw − denoised` is generated for each available subject/pipeline.

In [ ]:
def stage_paths(subject,pipeline):
    root=(V1_ROOT if pipeline=='v1' else V2_ROOT)/subject
    names={'raw':f'{subject}_01_dwi_raw.mif' if pipeline=='v1' else f'{subject}_01_raw.mif','denoised':f'{subject}_02_dwi_denoised.mif' if pipeline=='v1' else f'{subject}_02_denoised.mif','degibbs':f'{subject}_03_dwi_degibbs.mif' if pipeline=='v1' else f'{subject}_03_degibbs.mif','residual':f'{subject}_denoising_residual.mif'}
    return root,{k:(root/'qc'/v if k=='residual' else root/v) for k,v in names.items()}
denoise_rows=[]
for subject in SUBJECTS:
    for pipeline in ('v1','v2'):
        root,p=stage_paths(subject,pipeline); out=FREEVIEW_ROOT/subject; out.mkdir(parents=True,exist_ok=True)
        raw=out/f'{subject}_{pipeline}_raw_vol{DWI_VOLUME}.nii.gz'; den=out/f'{subject}_{pipeline}_denoised_vol{DWI_VOLUME}.nii.gz'
        if p['raw'].exists() and not raw.exists(): run_optional(['mrconvert',p['raw'],raw,'-coord','3',str(DWI_VOLUME),'-force'],raw)
        if p['denoised'].exists() and not den.exists(): run_optional(['mrconvert',p['denoised'],den,'-coord','3',str(DWI_VOLUME),'-force'],den)
        residual=out/f'{subject}_{pipeline}_denoising_residual_vol{DWI_VOLUME}.nii.gz'
        if raw.exists() and den.exists(): run_optional(['mrcalc',raw,den,'-subtract',residual,'-force'],residual)
        denoise_rows.append({'subject':subject,'pipeline':pipeline,'raw':'available' if p['raw'].exists() else 'missing','denoised':'available' if p['denoised'].exists() else 'missing','residual':'available' if residual.exists() else 'missing'})
denoise=pd.DataFrame(denoise_rows); denoise.to_csv(QC_ROOT/'denoising_qc_availability.csv',index=False); display(denoise)

## 4. Gibbs ringing QC

Denoised and degibbs images are compared separately from denoising. The missing visualization derivative `denoised − degibbs` is generated; inspect it near sharp tissue boundaries.

In [ ]:
gibbs_rows=[]
for subject in SUBJECTS:
    for pipeline in ('v1','v2'):
        root,p=stage_paths(subject,pipeline); out=FREEVIEW_ROOT/subject; den=out/f'{subject}_{pipeline}_denoised_vol{DWI_VOLUME}.nii.gz'; deg=out/f'{subject}_{pipeline}_degibbs_vol{DWI_VOLUME}.nii.gz'
        if p['degibbs'].exists() and not deg.exists(): run_optional(['mrconvert',p['degibbs'],deg,'-coord','3',str(DWI_VOLUME),'-force'],deg)
        residual=out/f'{subject}_{pipeline}_gibbs_residual_vol{DWI_VOLUME}.nii.gz'
        if den.exists() and deg.exists(): run_optional(['mrcalc',den,deg,'-subtract',residual,'-force'],residual)
        gibbs_rows.append({'subject':subject,'pipeline':pipeline,'denoised':'available' if den.exists() else 'missing','degibbs':'available' if deg.exists() else 'missing','gibbs_residual':'available' if residual.exists() else 'missing'})
gibbs=pd.DataFrame(gibbs_rows); gibbs.to_csv(QC_ROOT/'gibbs_qc_availability.csv',index=False); display(gibbs)

## 5. Eddy motion, outliers, and eddy_quad

This section runs after denoising and Gibbs QC, matching the preprocessing order. It summarizes RMS, writes translation/rotation curves, counts outliers by volume, and runs `eddy_quad` using preserved inputs when available.

In [ ]:
eddy_rows=[]; curve_rows=[]; outlier_rows=[]
for subject in SUBJECTS:
    for pipeline,root in [('v1',V1_ROOT/subject),('v2',V2_ROOT/subject)]:
        d=complete_eddy_dir(root); row={'subject':subject,'pipeline':pipeline,'status':'missing','scratch_dir':str(d) if d else ''}
        if d:
            row['status']='available'; motion=load_numbers(d/'dwi_post_eddy.eddy_movement_rms'); params=load_numbers(d/'dwi_post_eddy.eddy_parameters'); outliers=load_numbers(d/'dwi_post_eddy.eddy_outlier_map',1)
            if motion is not None: row['mean_absolute_rms']=float(np.mean(np.abs(motion))); row['maximum_relative_rms']=float(np.max(np.abs(motion)))
            if params is not None and params.shape[1]>=6:
                c=pd.DataFrame(params[:,:6],columns=['rotation_x','rotation_y','rotation_z','translation_x','translation_y','translation_z']); c.insert(0,'volume',np.arange(len(c))); c.insert(0,'pipeline',pipeline); c.insert(0,'subject',subject); c.to_csv(QC_ROOT/f'{subject}_{pipeline}_translation_rotation.csv',index=False); curve_rows.extend(c.to_dict('records'))
            if outliers is not None:
                counts=np.count_nonzero(outliers,axis=1); row['outlier_flagged']=int(counts.sum()); row['outlier_evaluated']=int(outliers.size); row['outlier_percent']=float(100*np.mean(outliers!=0)); outlier_rows.extend([{'subject':subject,'pipeline':pipeline,'volume':i,'outlier_slices':int(x),'outlier_percent':float(100*x/outliers.shape[1])} for i,x in enumerate(counts)])
            if RUN_EDDY_QUAD and shutil.which('eddy_quad'):
                report=REPORT_ROOT/subject/pipeline; report.mkdir(parents=True,exist_ok=True); cmd=['eddy_quad',str(d/'dwi_post_eddy'),'-idx',str(d/'eddy_indices.txt'),'-par',str(d/'eddy_config.txt'),'-m',str(d/'eddy_mask.nii'),'-b',str(d/'bvals'),'-o',str(report)]; r=subprocess.run(cmd,capture_output=True,text=True); row['eddy_quad_status']='completed' if r.returncode==0 else f'failed: {r.stderr[-300:]}'
            elif RUN_EDDY_QUAD: row['eddy_quad_status']='skipped: eddy_quad not found'
        eddy_rows.append(row)
eddy=pd.DataFrame(eddy_rows); eddy.to_csv(QC_ROOT/'eddy_qc_summary.csv',index=False); pd.DataFrame(curve_rows).to_csv(QC_ROOT/'translation_rotation_all_volumes.csv',index=False); pd.DataFrame(outlier_rows).to_csv(QC_ROOT/'outliers_by_volume.csv',index=False); display(eddy)

## 6. Corrected mean b0, brain masks, metrics, and regional stability

Existing corrected mean b0, masks, and FA/MD/AD/RD maps are read for QC. Only missing comparison derivatives are created. Regional statistics require an atlas already verified in DWI space and are skipped otherwise.

In [ ]:
availability=[]
for subject in SUBJECTS:
    for pipeline,root in [('v1',V1_ROOT/subject),('v2',V2_ROOT/subject)]:
        mask=root/f'{subject}_eddy_mask.nii.gz' if pipeline=='v1' else root/f'{subject}_corrected_mean_b0_brain_mask.nii.gz'
        for name,path in [('corrected_mean_b0',root/f'{subject}_corrected_mean_b0.nii.gz'),('mask',mask),('FA',root/f'{subject}_fa.nii.gz'),('MD',root/f'{subject}_md.nii.gz'),('AD',root/f'{subject}_ad.nii.gz'),('RD',root/f'{subject}_rd.nii.gz')]: availability.append({'subject':subject,'pipeline':pipeline,'product':name,'status':'available' if path.exists() else 'missing','path':str(path)})
pd.DataFrame(availability).to_csv(QC_ROOT/'b0_mask_metric_availability.csv',index=False); display(pd.DataFrame(availability))

## 7. Representative subjects and final deliverables

Use the Eddy summary to shortlist low-motion, typical, and high-motion subjects, then confirm the final choices visually using corrected mean b0, denoising residual, FA, and MD images.

In [ ]:
available=eddy[eddy.status=='available'].copy()
if not available.empty:
    s=available.groupby('subject').agg(mean_absolute_rms=('mean_absolute_rms','mean'),maximum_relative_rms=('maximum_relative_rms','mean'),outlier_percent=('outlier_percent','mean')).reset_index(); s['score']=s[['mean_absolute_rms','maximum_relative_rms','outlier_percent']].rank(pct=True).mean(axis=1); s=s.sort_values('score').reset_index(drop=True); reps=pd.DataFrame([{'role':'good / low motion',**s.iloc[0].to_dict()},{'role':'typical / medium motion',**s.iloc[len(s)//2].to_dict()},{'role':'higher motion / artifact review',**s.iloc[-1].to_dict()}])
else: reps=pd.DataFrame([{'role':'no complete Eddy data'}])
reps.to_csv(QC_ROOT/'representative_subjects.csv',index=False); display(reps)